In [26]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path("/home/fedor/Fast_Data_Discovery")
PAPER_ROOT = Path("/home/fedor/Matryoshka")
RUNTIMES_CSV = PROJECT_ROOT / "experiments/downstream/ablation/runtime/runtimes.csv"

df = pl.read_csv(RUNTIMES_CSV)
df = df.filter(pl.col('dataset') != 'elections')

# Rename "Ranking" -> "Sketching" (log files use the old name)
df = df.with_columns(
    pl.when(pl.col("step") == "Ranking")
    .then(pl.lit("Sketching"))
    .otherwise(pl.col("step"))
    .alias("step")
)

# Dataset ordering matching efficiency.tex (Regression | Classification)
DATASET_ORDER = [
    "nyc_realestate", "nyc_energy", "nyc_fire",
    "gittables_imdb", "cuk_jobs", "gittables_pageviews", "gittables_vgsales",
    "cuk_arrest", "gittables_food", "cuk_hospital", "nyc_trees",
]

# Lake symbol mapping: † = NYC, ‡ = CUK, * = GitTables
LAKE_SYMBOL = {
    "nyc_": r"$^\dagger$",
    "cuk_": r"$^\ddagger$",
    "gittables_": r"$^\star$",
}

def get_lake_symbol(ds: str) -> str:
    for prefix, sym in LAKE_SYMBOL.items():
        if ds.startswith(prefix):
            return sym
    return ""

DATASET_LABELS_BASE = [
    "Real Estate", "Energy", "Fire Incidents",
    "IMDB", "Jobs", "Page Views", "VG Sales",
    "Arrest", "Food", "Hospital", "Trees",
]
DATASET_LABELS = [
    f"{label}{get_lake_symbol(ds)}"
    for ds, label in zip(DATASET_ORDER, DATASET_LABELS_BASE)
]

STEPS_FORWARD = ["Retrieval", "Pruning", "Sketching", "Selection", "Augmentation"]

def build_step_rows(df: pl.DataFrame, strategy: str, steps: list[str]) -> dict[str, dict[str, float | None]]:
    """Return {step: {dataset: runtime}} for a given strategy."""
    sub = df.filter(
        (pl.col("strategy") == strategy) & (pl.col("step") != "Total")
    )
    result = {}
    for step in steps:
        step_data = sub.filter(pl.col("step") == step)
        row = {}
        for ds in DATASET_ORDER:
            val = step_data.filter(pl.col("dataset") == ds).select("runtime")
            row[ds] = val.item() if len(val) == 1 else None
        result[step] = row
    return result

fwd_rows = build_step_rows(df, "forward", STEPS_FORWARD)

# Build totals
fwd_totals = {}
for ds in DATASET_ORDER:
    fwd_totals[ds] = sum(v for v in (fwd_rows[s].get(ds) for s in STEPS_FORWARD) if v is not None)

print("Data loaded. Forward steps:", len(fwd_rows))

Data loaded. Forward steps: 5


In [27]:
def fmt(val, precision=3):
    """Format a runtime value for LaTeX."""
    if val is None:
        return "---"
    return f"{val:.{precision}f}"

def make_row(label, data, precision=3):
    """Build a single LaTeX table row."""
    cells = []
    for ds in DATASET_ORDER:
        cells.append(fmt(data.get(ds) if isinstance(data, dict) else data[ds], precision))
    reg = " & ".join(cells[:7])
    clf = " & ".join(cells[7:])
    return f"        {label}\n            & {reg}\n            & {clf} \\\\"

lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"    \small")
lines.append(r"    \centering")
lines.append(r"    \setlength\tabcolsep{3pt}")
lines.append(r"    \caption{Per-step runtime breakdown in seconds for Forward Selection. $^\dagger$\,NYC Open Data, $^\ddagger$\,CUK Open Data, $^\star$\,GitTables.}")
lines.append(r"    \vspace{-0.3cm}")
lines.append(r"")
lines.append(r"    \begin{tabular}{l|rrrrrrr|rrrr}")
lines.append(r"        \toprule")
lines.append(r"        & \multicolumn{7}{c|}{\textit{Regression}} & \multicolumn{4}{c}{\textit{Classification}} \\")
lines.append(r"        \cmidrule(lr){2-8} \cmidrule(l){9-12}")

# Column headers
hdr = "        \\textbf{Step}"
for label in DATASET_LABELS:
    hdr += f"\n            & \\textbf{{{label}}}"
hdr += " \\\\"
lines.append(hdr)

lines.append(r"        \midrule")
for step in STEPS_FORWARD:
    lines.append(make_row(step, fwd_rows[step]))
lines.append(r"        \cmidrule{1-12}")
lines.append(make_row(r"\textbf{Total}", fwd_totals))

lines.append(r"        \bottomrule")
lines.append(r"    \end{tabular}")
lines.append(r"")
lines.append(r"    \label{table:runtime_breakdown}")
lines.append(r"\end{table*}")

latex = "\n".join(lines)
print(latex)

\begin{table*}[t]
    \small
    \centering
    \setlength\tabcolsep{3pt}
    \caption{Per-step runtime breakdown in seconds for Forward Selection. $^\dagger$\,NYC Open Data, $^\ddagger$\,CUK Open Data, $^\star$\,GitTables.}
    \vspace{-0.3cm}

    \begin{tabular}{l|rrrrrrr|rrrr}
        \toprule
        & \multicolumn{7}{c|}{\textit{Regression}} & \multicolumn{4}{c}{\textit{Classification}} \\
        \cmidrule(lr){2-8} \cmidrule(l){9-12}
        \textbf{Step}
            & \textbf{Real Estate$^\dagger$}
            & \textbf{Energy$^\dagger$}
            & \textbf{Fire Incidents$^\dagger$}
            & \textbf{IMDB$^\star$}
            & \textbf{Jobs$^\ddagger$}
            & \textbf{Page Views$^\star$}
            & \textbf{VG Sales$^\star$}
            & \textbf{Arrest$^\ddagger$}
            & \textbf{Food$^\star$}
            & \textbf{Hospital$^\ddagger$}
            & \textbf{Trees$^\dagger$} \\
        \midrule
        Retrieval
            & 5.657 & 3.392 & 1.958 & 75.330 &

In [28]:
# Save to Matryoshka/tables
out_path = PAPER_ROOT / "tables" / "runtime_breakdown.tex"
out_path.write_text(latex)
print(f"Saved to {out_path}")

Saved to /home/fedor/Matryoshka/tables/runtime_breakdown.tex
